# HUGGE — пакетная генерация TRELLIS.2 Q8

Продолжает генерацию с 7-го товара: 6 готовых моделей уже сохранены локально. После сбоя автоматически перезапускает TRELLIS.2, повторяет текущий товар и скачивает каждый готовый PBR GLB сразу. Требуется **T4 GPU**.

In [ ]:
import os, pathlib, subprocess, time, tarfile, requests, socket, shutil, json

base = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
work = base / 'trellis2'
outputs = base / 'hugge-trellis2-output'
work.mkdir(parents=True, exist_ok=True)
outputs.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)

def remote_size(url):
    response = requests.head(url, allow_redirects=True, timeout=60)
    response.raise_for_status()
    return int(response.headers.get('content-length', 0))

def download_resume(url, destination, retries=8):
    destination = pathlib.Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = remote_size(url)
    if destination.exists() and expected and destination.stat().st_size == expected:
        print(f'✓ {destination.name} уже загружен')
        return
    if destination.exists() and expected and destination.stat().st_size > expected:
        destination.unlink()
    for attempt in range(1, retries + 1):
        offset = destination.stat().st_size if destination.exists() else 0
        headers = {'Range': f'bytes={offset}-'} if offset else {}
        try:
            with requests.get(url, headers=headers, stream=True, allow_redirects=True, timeout=(60, 300)) as response:
                if response.status_code == 416 and expected == offset:
                    return
                response.raise_for_status()
                append = offset > 0 and response.status_code == 206
                mode = 'ab' if append else 'wb'
                if not append:
                    offset = 0
                downloaded = offset
                last_report = downloaded
                with destination.open(mode) as target:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        if chunk:
                            target.write(chunk)
                            downloaded += len(chunk)
                            if downloaded - last_report >= 512 * 1024 * 1024:
                                print(f'  {destination.name}: {downloaded / 1024**3:.1f} / {expected / 1024**3:.1f} ГБ')
                                last_report = downloaded
            if not expected or destination.stat().st_size == expected:
                print(f'✓ {destination.name}')
                return
            raise IOError(f'неполный файл: {destination.stat().st_size} из {expected}')
        except Exception as error:
            if attempt == retries:
                raise
            print(f'Повтор {attempt}/{retries}: {error}')
            time.sleep(min(30, attempt * 3))

runtime = work / 'runtime'
server_bin = runtime / 'trellis-server'
if not server_bin.exists():
    compute_cap = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True).strip().splitlines()[0]
    backend = 'cuda12' if float(compute_cap) < 7.5 else 'cuda'
    archive = work / f'trellis-{backend}-linux-x64.tar.gz'
    download_resume(f'https://github.com/pwilkin/trellis.cpp/releases/latest/download/trellis-{backend}-linux-x64.tar.gz', archive)
    runtime.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(runtime)
    server_bin.chmod(0o755)

models = work / 'models'
model_names = ['birefnet.gguf', 'dinov3.gguf', 'ss_flow.gguf', 'ss_dec.gguf', 'shape_flow_512.gguf', 'shape_flow_1024.gguf', 'shape_dec.gguf', 'tex_flow_512.gguf', 'tex_flow_1024.gguf', 'tex_dec.gguf']
for model_name in model_names:
    download_resume(f'https://huggingface.co/ilintar/trellis2-gguf/resolve/main/q8/{model_name}', models / model_name)
print('TRELLIS.2 runtime and Q8 weights are ready')

server = None
server_url = None
server_log = None

def stop_server():
    global server, server_log
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=20)
        except subprocess.TimeoutExpired:
            server.kill()
    if server_log is not None:
        server_log.close()
    server = None
    server_log = None

def start_server():
    global server, server_url, server_log
    stop_server()
    with socket.socket() as free_socket:
        free_socket.bind(('127.0.0.1', 0))
        server_port = free_socket.getsockname()[1]
    server_url = f'http://127.0.0.1:{server_port}'
    env = os.environ.copy()
    env['LD_LIBRARY_PATH'] = f"{runtime}:{env.get('LD_LIBRARY_PATH', '')}"
    log_path = work / f'trellis-server-{server_port}.log'
    server_log = open(log_path, 'w')
    server = subprocess.Popen([str(server_bin), '--models', str(models), '--host', '127.0.0.1', '--port', str(server_port), '--res', '1024', '--require-gpu'], stdout=server_log, stderr=subprocess.STDOUT, env=env)
    for _ in range(180):
        try:
            if requests.get(f'{server_url}/health', timeout=2).ok:
                print(f'TRELLIS.2 server ready: {server_url}')
                return
        except requests.RequestException:
            pass
        if server.poll() is not None:
            raise RuntimeError(log_path.read_text()[-5000:])
        time.sleep(2)
    raise TimeoutError('TRELLIS server did not become ready')

start_server()

products = [
    ('102923', 'Тумба Polo белая'), ('100489', 'Стол Roxby'), ('98600', 'Стол Montreux'),
    ('90157', 'Стул Brooke зелёный'), ('89099', 'Стул Brooke серый'), ('71939', 'Стул Cazar'),
    ('35348', 'Стол Writex'), ('90315', 'Тумба Polo чёрная'), ('85345', 'Стол Neptun'),
]
site = 'https://mirrai-try-on.moonlight-5782.chatgpt.site'
progress_path = outputs / 'progress.json'
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
for index, (sku, name) in enumerate(products, 7):
    output = outputs / f'hugge-{sku}-trellis2-q8-pbr.glb'
    if output.exists() and output.stat().st_size > 100000:
        print(f'[{index}/15] ✓ {name} уже готов')
        progress[sku] = {'status': 'ready', 'file': output.name, 'bytes': output.stat().st_size}
        progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2))
        continue
    image_response = requests.get(f'{site}/catalog-sources/hugge-md/{sku}-1.jpg', timeout=120)
    image_response.raise_for_status()
    for attempt in range(1, 4):
        print(f'[{index}/15] Генерация: {name} ({sku}), попытка {attempt}/3')
        try:
            response = requests.post(
                f'{server_url}/generate',
                files={'image': (f'{sku}.jpg', image_response.content, 'image/jpeg')},
                data={'seed': sku, 'resolution': '1024', 'bg_removal': 'birefnet'},
                timeout=3600,
            )
            if not response.ok:
                raise RuntimeError(f'HTTP {response.status_code}: {response.text[:2000]}')
            output.write_bytes(response.content)
            if output.stat().st_size < 100000:
                raise RuntimeError(f'слишком маленький GLB: {output.stat().st_size} байт')
            progress[sku] = {'status': 'ready', 'file': output.name, 'bytes': output.stat().st_size}
            print(f'[{index}/15] ✓ {output.name}: {output.stat().st_size / 1024**2:.1f} МБ')
            progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2))
            from google.colab import files
            files.download(str(output))
            break
        except Exception as error:
            progress[sku] = {'status': 'retrying' if attempt < 3 else 'failed', 'error': str(error)}
            progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2))
            print(f'[{index}/15] ✗ попытка {attempt}: {error}')
            if attempt < 3:
                print('Перезапускаю TRELLIS.2 и повторяю этот товар…')
                start_server()
    progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2))

zip_path = pathlib.Path(shutil.make_archive(str(base / 'hugge-trellis2-models'), 'zip', outputs))
ready = sum(item.get('status') == 'ready' for item in progress.values())
print(f'Очередь завершена: {ready}/15 моделей. Архив: {zip_path}')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))